![GovSpace Logo](https://govspace.io/wp-content/uploads/2025/09/GovSpace-web.svg)

# 🎛️ Graduated Autonomy: Human-in-the-Loop Agents

> **⚠️ Before you begin:** Copy any notebook you want to edit into your personal folder (e.g. `matt-grasser/`) before making changes. The `TEMPLATES` folder syncs from GitHub on each login and is read-only — working in your own folder keeps your work safe. Your folder was created automatically when you logged in.

In production, you rarely want a fully autonomous agent. Instead, you want **graduated autonomy** — a spectrum from full human control to full agent autonomy, with checkpoints in between.

## The Autonomy Spectrum

| Level | Description | Example |
|-------|-------------|--------|
| **Manual** | Human does everything, AI assists | AI drafts, human edits and sends |
| **Supervised** | AI proposes actions, human approves | AI suggests response, human clicks 'send' |
| **Guided** | AI acts, human reviews periodically | AI handles routine tasks, flags edge cases |
| **Autonomous** | AI acts independently within guardrails | AI processes and responds within defined rules |

## What you'll learn
1. LangGraph interrupt/checkpoint patterns
2. Building approval gates into agent workflows
3. Dynamic autonomy levels based on confidence
4. How to build trust incrementally

In [ ]:
import os

# Keys should already be set from 00-Setup. If not, set them here:
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

from typing import Annotated, TypedDict, Literal
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

model = ChatAnthropic(model="claude-sonnet-4-20250514", max_tokens=1024)

## Pattern 1: Approval Gate

The simplest human-in-the-loop pattern: the agent proposes an action, and a human approves or rejects it before execution.

```
[Draft] → [Human Review] → [Execute] → END
               ↓
           [Revise] → [Human Review]  (loop)
```

In [ ]:
class EmailState(TypedDict):
    """State for an email drafting workflow."""
    request: str
    messages: Annotated[list, add_messages]
    draft: str
    feedback: str
    approved: bool
    revision_count: int


def draft_email(state: EmailState) -> dict:
    """AI drafts (or revises) an email."""
    revision = state.get("revision_count", 0)

    if revision > 0 and state.get("feedback"):
        prompt = (
            f"Revise this email draft based on feedback.\n\n"
            f"Original request: {state['request']}\n\n"
            f"Current draft:\n{state['draft']}\n\n"
            f"Feedback: {state['feedback']}\n\n"
            f"Write the revised email only, no commentary."
        )
    else:
        prompt = (
            f"Draft a professional email for: {state['request']}\n\n"
            f"Write the email only, no commentary."
        )

    response = model.invoke([HumanMessage(content=prompt)])
    print(f"\n{'📝 Draft' if revision == 0 else f'✏️ Revision {revision}'}:")
    print("-" * 40)
    print(response.content)
    print("-" * 40)

    return {
        "draft": response.content,
        "messages": [response],
        "revision_count": revision + 1,
    }


def human_review(state: EmailState) -> dict:
    """Simulate human review (in production, this would pause for real input)."""
    print("\n🧑 HUMAN REVIEW")
    print("Options: [a]pprove, [r]evise with feedback, [c]ancel")

    # In a notebook, we use input() for interactive review
    choice = input("Your choice: ").strip().lower()

    if choice == "a":
        print("✅ Approved!")
        return {"approved": True, "feedback": ""}
    elif choice.startswith("r"):
        feedback = input("Feedback: ").strip()
        print(f"🔄 Sending back for revision: {feedback}")
        return {"approved": False, "feedback": feedback}
    else:
        print("❌ Cancelled")
        return {"approved": True, "draft": "[CANCELLED]", "feedback": ""}


def execute_send(state: EmailState) -> dict:
    """Execute the approved action (send email)."""
    if state.get("draft") == "[CANCELLED]":
        print("\n🚫 Email cancelled, not sent.")
    else:
        print(f"\n📤 Email sent! (after {state.get('revision_count', 0)} revision(s))")
    return state


def review_router(state: EmailState) -> str:
    """Route based on human review decision."""
    if state.get("approved", False):
        return "execute"
    return "revise"

In [ ]:
# Build the approval gate workflow
workflow = StateGraph(EmailState)

workflow.add_node("draft", draft_email)
workflow.add_node("review", human_review)
workflow.add_node("execute", execute_send)

workflow.set_entry_point("draft")
workflow.add_edge("draft", "review")
workflow.add_conditional_edges(
    "review",
    review_router,
    {"execute": "execute", "revise": "draft"},
)
workflow.add_edge("execute", END)

email_app = workflow.compile()
print("Email workflow compiled!")

In [ ]:
# Run it! You'll be prompted to approve/revise the draft.
result = email_app.invoke({
    "request": "Invite the AI Builders Lab participants to our March 18 session. "
               "Mention it's a hands-on collaborative session using Jupyter notebooks "
               "with pre-installed AI tools. Keep it short and enthusiastic.",
    "messages": [],
    "draft": "",
    "feedback": "",
    "approved": False,
    "revision_count": 0,
})

## Pattern 2: Confidence-Based Autonomy

Instead of always asking for approval, the agent self-assesses its confidence and only asks for review when uncertain.

```
[Classify] → [High confidence] → [Auto-execute] → END
                ↓
           [Low confidence] → [Human Review] → [Execute] → END
```

In [ ]:
import json


class TaskState(TypedDict):
    """State for confidence-based routing."""
    task: str
    messages: Annotated[list, add_messages]
    classification: str
    confidence: float
    response: str
    autonomy_level: str  # "auto" or "review"


def classify_task(state: TaskState) -> dict:
    """Classify the task and assess confidence."""
    prompt = (
        f"Classify this task and rate your confidence in handling it.\n\n"
        f"Task: {state['task']}\n\n"
        f"Respond in JSON format:\n"
        f'{{"classification": "routine|complex|sensitive", '
        f'"confidence": 0.0-1.0, '
        f'"reasoning": "brief explanation"}}'
    )

    response = model.invoke([HumanMessage(content=prompt)])

    try:
        # Extract JSON from response
        text = response.content
        # Handle markdown code blocks
        if "```" in text:
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        parsed = json.loads(text.strip())
    except (json.JSONDecodeError, IndexError):
        parsed = {"classification": "complex", "confidence": 0.5, "reasoning": "Could not parse"}

    classification = parsed.get("classification", "complex")
    confidence = float(parsed.get("confidence", 0.5))

    # Determine autonomy level based on confidence + classification
    if classification == "routine" and confidence >= 0.8:
        autonomy = "auto"
    elif classification == "sensitive":
        autonomy = "review"  # Always review sensitive tasks
    elif confidence >= 0.9:
        autonomy = "auto"
    else:
        autonomy = "review"

    print(f"📊 Classification: {classification} | Confidence: {confidence:.0%} | → {autonomy}")
    print(f"   Reasoning: {parsed.get('reasoning', 'N/A')}")

    return {
        "classification": classification,
        "confidence": confidence,
        "autonomy_level": autonomy,
        "messages": [response],
    }


def generate_response(state: TaskState) -> dict:
    """Generate a response to the task."""
    prompt = f"Handle this task concisely:\n\n{state['task']}"
    response = model.invoke([HumanMessage(content=prompt)])

    if state["autonomy_level"] == "auto":
        print(f"\n🤖 Auto-executing (confidence was high):")
    else:
        print(f"\n📋 Proposed response (awaiting review):")

    print(response.content[:300] + ("..." if len(response.content) > 300 else ""))

    return {"response": response.content, "messages": [response]}


def human_checkpoint(state: TaskState) -> dict:
    """Human reviews the proposed response."""
    print("\n🧑 HUMAN CHECKPOINT — This task was flagged for review.")
    choice = input("[a]pprove / [r]eject: ").strip().lower()

    if choice == "a":
        print("✅ Approved")
    else:
        print("❌ Rejected — response discarded")
        return {"response": "[REJECTED BY HUMAN]"}

    return state


def autonomy_router(state: TaskState) -> str:
    """Route based on autonomy level."""
    return state.get("autonomy_level", "review")

In [ ]:
# Build the confidence-based workflow
workflow2 = StateGraph(TaskState)

workflow2.add_node("classify", classify_task)
workflow2.add_node("generate", generate_response)
workflow2.add_node("checkpoint", human_checkpoint)

workflow2.set_entry_point("classify")
workflow2.add_edge("classify", "generate")
workflow2.add_conditional_edges(
    "generate",
    autonomy_router,
    {"auto": END, "review": "checkpoint"},
)
workflow2.add_edge("checkpoint", END)

autonomy_app = workflow2.compile()
print("Confidence-based autonomy workflow compiled!")

In [ ]:
# Test with a ROUTINE task (should auto-execute)
print("=" * 50)
print("Test 1: Routine task")
print("=" * 50)
autonomy_app.invoke({
    "task": "What is 2 + 2?",
    "messages": [], "classification": "", "confidence": 0.0,
    "response": "", "autonomy_level": "",
})

In [ ]:
# Test with a SENSITIVE task (should require review)
print("=" * 50)
print("Test 2: Sensitive task")
print("=" * 50)
autonomy_app.invoke({
    "task": "Draft a public statement about our company's data breach incident",
    "messages": [], "classification": "", "confidence": 0.0,
    "response": "", "autonomy_level": "",
})

## 💡 Key Takeaways

1. **Start supervised, earn autonomy** — Begin with human approval for everything, then relax controls as trust builds
2. **Classify before acting** — Let the agent assess task complexity/sensitivity to determine the right autonomy level
3. **Always keep an escape hatch** — Even autonomous agents should have human override capability
4. **Log everything** — Every decision, every approval/rejection, for audit and improvement

## Design Principles for Production

- **Sensitive actions always require review** (data deletion, public communications, financial transactions)
- **Confidence thresholds are tunable** — start conservative (0.95), lower as the system proves reliable
- **Human feedback improves the system** — track rejection reasons to improve classification
- **Autonomy is per-task, not per-agent** — the same agent can be autonomous for routine tasks and supervised for sensitive ones

---

<div style="background-color:#1dc28b; color:white; padding:14px; border-radius:5px; margin-bottom:10px;">
<strong>🎉 Congratulations!</strong> You've completed the GovSpace Agentic Gym starter notebooks. You now have the building blocks for:
<ul style="margin-top:8px; margin-bottom:0;">
<li>Tool-calling agents (01)</li>
<li>Stateful multi-step workflows (02)</li>
<li>Human-in-the-loop graduated autonomy (03)</li>
</ul>
</div>

**What to build next:** Try combining these patterns — a LangGraph workflow with tools AND human checkpoints at critical decision points.

Explore more at [GovSpace Connect](https://app.govspace.io/connect) and [GovSpace Discover](https://app.govspace.io/discover/library?view=network).